# 1. The Core Objective
In your current BiLSTM, you have a sequence of hidden states $H = \{h_1, h_2, \ldots, h_T\}$, where each $h_t \in \mathbb{R}^{2 \times \text{hidden_dim}}$.  
Pooling assumes we know which $h_t$ are important (all of them equally, or just the maximum).  
Attention assumes the importance of $h_t$ is a latent variable we need to learn during backpropagation.

# 2. Mathematical Breakdown (The MLP Approach)
Since you are not using a Transformer, you are implementing Additive (Bahdanau-style) Attention. It follows three mathematical steps:

## 2.1 Step A: Scoring ($e_t$)
We pass each hidden state through a small alignment network (a Linear layer) to get a scalar importance score.
$$ e_t = \mathbf{v}^\top \tanh(\mathbf{W} h_t + \mathbf{b}) $$
- $\mathbf{W}$: Weight matrix that learns which features in the $2H$ vector signal "sentiment."
- $\tanh$: Non-linearity that allows the model to capture complex, non-obvious importance.
- $\mathbf{v}$: A weight vector that collapses the hidden representation into a single score.

## 2.2 Step B: Normalization ($\alpha_t$)
To ensure the weights are comparable across different sentence lengths, we use Softmax.
$$ \alpha_t = \frac{\exp(e_t)}{\sum_{i=1}^{T} \exp(e_i)} $$
This ensures $\sum_t \alpha_t = 1$. It turns raw scores into a probability distribution of "where to look."

## 2.3 Step C: The Context Vector ($c$)
The final representation is a weighted sum of the LSTM outputs.
$$ c = \sum_{t=1}^{T} \alpha_t h_t $$
If $h_{\text{"terrible"}}$ has a high $\alpha$, the final vector $c$ will look almost exactly like the "terrible" vector.

# 3. Code Implementation (Additive Attention)
In your Model2, replace your pooling logic with this:

```python
# --- Inside __init__ ---
self.attn_linear = torch.nn.Linear(2 * hidden_dim, 2 * hidden_dim)
self.v = torch.nn.Linear(2 * hidden_dim, 1, bias=False)
self.forward_layer1 = torch.nn.Linear(2 * hidden_dim, output_dim)

# --- Inside forward ---
# out shape: [batch, seq_len, 2 * hidden_dim]
energy = torch.tanh(self.attn_linear(out))  # [batch, seq_len, 2H]
scores = self.v(energy)                     # [batch, seq_len, 1]
# Apply mask to ignore padding tokens
scores[mask == 0] = -float('inf')

attn_weights = torch.softmax(scores, dim=1)  # [batch, seq_len, 1]
context = torch.sum(out * attn_weights, dim=1)  # [batch, 2H]
output = self.forward_layer1(self.dropout(context))
```

# 4. Key Differences: Your Model vs. Transformer
| Feature | Your Additive Attention | Transformer Self-Attention |
|---|---|---|
| Formula | $v^\top \tanh(W h)$ | $\text{softmax}\left(\frac{Q K^\top}{\sqrt{d}}\right) V$ |
| Comparison | Independent: Scores each word alone. | Relational: Words compare to each other. |
| Logic | "Is this word important?" | "How much does word A relate to word B?" |
| Math | Uses a Linear Layer to score. | Uses Dot-Product of vectors to score. |

# 5. Why this fixes "Stagnant" Accuracy
- **Learned Pooling**: Instead of mean pooling (which dilutes important words), the model learns importance weights.
- **Cleaner Signal**: The final vector becomes a weighted, high-signal representation instead of a noisy average.

# 6. Linear (Dot-Product Style) Attention
In this version, you learn a single "Template Vector" ($w$) representing the "Ideal Sentiment Word."

## 6.1 The Logic
- The Weight: A fixed weight vector $w \in \mathbb{R}^{2H}$.
- The Scoring: For every word $h_t$, calculate the dot product $w \cdot h_t$.
- The Result: If $h_t$ aligns with $w$ (strong sentiment), the score is high.

## 6.2 Code Implementation
```python
# --- Inside __init__ ---
self.attn_weights_layer = torch.nn.Linear(2 * hidden_dim, 1, bias=False)
self.forward_layer1 = torch.nn.Linear(2 * hidden_dim, output_dim)

# --- Inside forward ---
scores = self.attn_weights_layer(out)  # [batch, seq_len, 1]
scores[mask == 0] = -float('inf')

attn_weights = torch.softmax(scores, dim=1)
context = torch.sum(out * attn_weights, dim=1)  # [batch, 2H]
output = self.forward_layer1(self.dropout(context))
```

# 7. Comparing Pooling Strategies

| Method | How it handles $h_t$ | Math Equivalent |
|---|---|---|
| Mean Pooling | Everyone is equal | $$ \frac{1}{T} \sum_{t=1}^{T} h_t $$ |
| Max Pooling | Only the loudest wins | $$ \max_{t} h_t $$ |
| Linear Attention | Learned importance per word | $$ \sum_{t=1}^T \text{softmax}(w \cdot h_t) h_t $$ |

# Handling Imbalance in Sports and Outdoor Dataset

## 1. **Loss Weighting**
- Assign higher weights to rare classes → bigger penalties for mistakes
- Assign lower weights to frequent classes → smaller penalties
- **Effect:** Forces the model to pay more attention to minority classes

## 2. **Sampling Weighting**
- Sample rare classes more often
- Sample frequent classes less often
- **Effect:** Provides the model with more balanced batches

---

## 3. **Key Conceptual Difference**
- **Loss weighting:** The model sees the *same* imbalanced data, but mistakes on rare classes hurt more.
- **Sampling weighting:** The model sees *more balanced* data, naturally learning minority patterns better.


## 4. **Why Sampling Weighting Is More Effective**
- **Intuition:**  
  - Loss weighting: The data input remains imbalanced; only penalties change.  
  - Sampling weighting: The data input becomes more balanced, so the model learns minority patterns directly.

- **Deeper insight:**  
  - Sampling changes *what data the model learns from*.  
  - Loss weighting only changes *how errors are penalized* after data is seen.

- **Analogy:**  
  - Loss weighting: Teacher emphasizes mistakes  
  - Sampling weighting: Teacher provides more practice on weak topics

> **Long-term benefit:** Practice (sampling) is generally more effective.

---

## 5. **Practitioners’ Approach**
- Combine both:  
  - Use `WeightedRandomSampler` + class-weighted loss

---

## 6. **Implementation Step**
  - Sampling happens *before* the model sees data, inside the Dataset → Sampler → DataLoader pipeline.

---

## 7. **Adjusting DataLoader**
- If you’re using a sampler, **remove** `shuffle=True` because the sampler handles randomness.

```python
train_loader = DataLoader(dataset, batch_size=..., sampler=sampler, shuffle=False)
```

---

## 8. **Creating Sample Weights**
- Use class counts:  
  ```python
  class_counts = data['overall'].value_counts()
  ```
- Assign weight per class:  
  ```python
  class_weight = {c: 1.0 / freq for c, freq in class_counts.items()}
  ```
- Assign sample weights based on class label:  
  ```python
  sample_weights = [class_weight[label] for label in dataset_labels]
  ```

---

## 9. **Using WeightedRandomSampler**
- **Set number of samples:**  
  - **Correct choice:** `len(dataset)`  
  - **Why?** To balance data without losing information.  
  ```python
  sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
  ```

- **Final DataLoader setup:**  
  ```python
  train_loader = DataLoader(dataset, batch_size=..., sampler=sampler, shuffle=False)
  ```

---

## 10. **Why `replacement=True`?**
- Ensures minority class samples are repeated enough times to balance classes.
- **Answer:** Because minority classes are sampled *with replacement* to appear more often.

---

## 11. **Should you keep class weights in the loss?**
- **Initially:** *No*, because sampling already balances class exposure.
- **Reason:**  
  - Sampling changes *data distribution*  
  - Loss weights only *penalize errors*, risking overcorrection if combined with sampling.

> **Best practice:**  
> - Use **sampling** (WeightedRandomSampler)  
> - Do **not** use class weights in loss initially

---

## 12. **Summary**
- Use **sampling weighting** in DataLoader for better minority class exposure.
- Remove `shuffle=True` when using sampler.
- Keep `weight=...` in `CrossEntropyLoss` off unless fine-tuning.
- This approach aligns *training data* with *model learning*, leading to more balanced performance.


In [1]:
import torch
import pandas as pd
import regex as re
from collections import Counter
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data = pd.read_json("/content/drive/MyDrive/Sports_and_Outdoors_5.json",lines=True)
data.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,AIXZKN4ACSKI,1881509818,David Briner,"[0, 0]",This came in on time and I am veru happy with ...,5,Woks very good,1390694400,"01 26, 2014"
1,A1L5P841VIO02V,1881509818,Jason A. Kramer,"[1, 1]",I had a factory Glock tool that I was using fo...,5,Works as well as the factory tool,1328140800,"02 2, 2012"
2,AB2W04NI4OEAD,1881509818,J. Fernald,"[2, 2]",If you don't have a 3/32 punch or would like t...,4,"It's a punch, that's all.",1330387200,"02 28, 2012"
3,A148SVSWKTJKU6,1881509818,"Jusitn A. Watts ""Maverick9614""","[0, 0]",This works no better than any 3/32 punch you w...,4,It's a punch with a Glock logo.,1328400000,"02 5, 2012"
4,AAAWJ6LW9WMOO,1881509818,Material Man,"[0, 0]",I purchased this thinking maybe I need a speci...,4,"Ok,tool does what a regular punch does.",1366675200,"04 23, 2013"


In [4]:
def preprocess(text):
  return re.sub(r'[^a-z]'," ",text.lower()).split()

data['reviewText'] = data['reviewText'].apply(lambda x: preprocess(x))

In [5]:
# Next step is to Build Vocabulary and we use collections module but we need to flatten all lists first

In [6]:
def flatten(dat):
  a = []
  for i in dat:
    a.extend(i)
  return a

flattened_tokens = flatten(data['reviewText'].to_list())

In [7]:
word_freq = Counter(flattened_tokens)

In [8]:
def filter(min_count,word_freq):
  dict = {}
  for i in word_freq.items():
    if i[1] >= min_count:
      dict[i[0]] = i[1]
  return dict

In [9]:
filtered_word_freq = filter(5,word_freq)
print(len(word_freq))
print(len(filtered_word_freq))

109352
31603


In [10]:
def build_vocab(wordfrequency):
  vocab = {}
  vocab['PAD'] = 0
  vocab['UNK'] = 1
  for i,j in enumerate(sorted(wordfrequency.items(), key = lambda x : x[1] , reverse=True ) ):
    vocab[j[0]] = i+2
  return vocab

vocabulary = build_vocab(filtered_word_freq)

In [11]:
def sentence_to_num(data,vocab):
  super_list = []
  for i in data.tolist():
    minor_list = []
    for j in i:
      if j in vocab.keys():
        minor_list.append(vocab[j])
      else:
        minor_list.append(vocab['UNK'])
    super_list.append(minor_list)
  return super_list

In [12]:
encoded_sentences_notpadded = sentence_to_num(data['reviewText'],vocabulary)

In [13]:
def max_len(sen):
  t = 0
  for i,j in enumerate(sen):
    if len(j)>t:
      t = len(j)
  return t,i+1

max,num_sentences = max_len(encoded_sentences_notpadded)
print("Maximum length of Sentence  ",max,"  Number of sentences  ",num_sentences)

Maximum length of Sentence   5773   Number of sentences   296337


In [14]:
# Using Numpy array to lower the RAM Usage
def padding(sen,maxlen):

  temp = np.zeros((num_sentences,maxlen),dtype = 'int32' )
  for i,j in enumerate(sen):
    if len(j) < maxlen:
      temp[i,:len(j)] = j
  return temp

padded_sentence = padding(encoded_sentences_notpadded,200)  #Custom Max_len to reduce RAM Usage
padded_sentence.shape

(296337, 200)

In [15]:
def sentiment(x):
  label = []
  count_1 = count_2 = count_3 = 0
  for i in x.to_list():
    if i==1 or i==2:
      label.append(1)
      count_1 += 1
    elif i==3:
      label.append(2)
      count_2 += 1
    else:
      label.append(3)
      count_3 += 1
  return label,count_1,count_2,count_3


label,one,two,three = sentiment(data['overall'])
label = np.array(label)
class_weight = torch.tensor([1.0/one,1.0/two,1.0/three],dtype = torch.float)
print(one,two,three)

19249 24071 253017


In [16]:
# We have to first split the data , and use TensorDataset which samples the data as x,y pairs(useful for iterating and batching,shuffling)

index = np.arange(0,num_sentences)
np.random.shuffle(index)  #Shuffling the indices before splitting

from torch.utils.data import TensorDataset,DataLoader,WeightedRandomSampler

#This is for directly predicting the rating 1,2,3,4,5
#train_data = TensorDataset(torch.tensor(padded_sentence[index[0:int(.7*num_sentences)]]) , torch.tensor(label.iloc[index[0:int(.7*num_sentences)]].to_numpy()))
#valid_data = TensorDataset(torch.tensor(padded_sentence[index[int(.7*num_sentences):int(.85*num_sentences)]]) , torch.tensor(label.iloc[index[int(.7*num_sentences):int(.85*num_sentences)]].to_numpy()))
#test_data = TensorDataset(torch.tensor(padded_sentence[index[int(.85*num_sentences):num_sentences]]) , torch.tensor(label.iloc[index[int(.85*num_sentences):num_sentences]].to_numpy()))

train_data = TensorDataset(torch.tensor(padded_sentence[index[0:int(.7*num_sentences)]]) , torch.tensor(label[index[0:int(.7*num_sentences)]]))
valid_data = TensorDataset(torch.tensor(padded_sentence[index[int(.7*num_sentences):int(.85*num_sentences)]]) , torch.tensor(label[index[int(.7*num_sentences):int(.85*num_sentences)]]))
test_data = TensorDataset(torch.tensor(padded_sentence[index[int(.85*num_sentences):num_sentences]]) , torch.tensor(label[index[int(.85*num_sentences):num_sentences]]))

train_labels = label[index[0:int(.7*num_sentences)]]
valid_labels = label[index[int(.7*num_sentences):int(.85*num_sentences)]]
test_labels = label[index[int(.85*num_sentences):num_sentences]]

train_weight = torch.tensor([class_weight[i-1] for i in train_labels],dtype = torch.float)

train_sample = WeightedRandomSampler(train_weight , len(train_weight) , replacement = True )

"""valid_weight = torch.tensor([class_weight[i-1] for i in valid_labels],dtype = torch.float)

valid_sample = WeightedRandomSampler(valid_weight , len(valid_weight) , replacement = True )

test_weight = torch.tensor([class_weight[i-1] for i in test_labels],dtype = torch.float)

test_sample = WeightedRandomSampler(test_weight , len(test_weight) , replacement = True )"""

# ONLY train uses sample valid/test → normal DataLoader because they should represent real world data distribution

'valid_weight = torch.tensor([class_weight[i-1] for i in valid_labels],dtype = torch.float)\n\nvalid_sample = WeightedRandomSampler(valid_weight , len(valid_weight) , replacement = True )\n\ntest_weight = torch.tensor([class_weight[i-1] for i in test_labels],dtype = torch.float)\n\ntest_sample = WeightedRandomSampler(test_weight , len(test_weight) , replacement = True )'

In [17]:
# Now , we have to shuffle the data and create batches(similar to batch_size in tensorflow) using Dataloader

train_loader = DataLoader(train_data , shuffle = True , batch_size= 60 )
valid_loader = DataLoader(valid_data , shuffle = True , batch_size= 60 )
test_loader = DataLoader(test_data , shuffle = True , batch_size= 60 )



In [18]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 80.9 MB/s eta 0:00:00


In [19]:
from gensim.models import Word2Vec

In [20]:
from google.colab import drive

#model = Word2Vec(data['reviewText'].to_list(),vector_size = 200 , sg = 1 , workers = 4 ,window = 5 ,min_count = 5 , epochs = 5)
model = Word2Vec.load("/content/drive/MyDrive/word2vec_sports.model")

In [21]:
# making the embedding matrix

embedded_matrix = torch.zeros((len(vocabulary),model.vector_size),dtype = torch.float32) # Embedding matrix is word and it's corresponding vector

for i in vocabulary:
  if i in model.wv:
    a = torch.from_numpy(model.wv[i].copy())
    embedded_matrix[vocabulary[i]] = a

# Building the actual model

In [22]:
from torch._C import device
from torch.nn.utils.rnn import pack_padded_sequence , pad_packed_sequence
class Model2(torch.nn.Module):
  def __init__(self, vocab_size , embedding_dim ,hidden_dim ,num_layers, output_dim ,embedding_matrix) -> None:
    super().__init__()
    self.embedding = torch.nn.Embedding(vocab_size , embedding_dim , padding_idx=0) # O/P is (batches,words_in_each_sen,embedding_dim)
    self.embedding.weight.data.copy_(embedding_matrix)
    self.embedding.weight.requires_grad = False # Already checked with False , val_accuracy reached 68.5 plateau and with True 67%
    self.dropout = torch.nn.Dropout(0.4)
    self.forward_layer1 = torch.nn.Linear(2*hidden_dim,output_dim)
    self.num_layers = num_layers
    self.hidden_dim = hidden_dim
    self.v = torch.nn.Linear(2*hidden_dim,1)
    self.attention = torch.nn.Linear(2 * hidden_dim, 2 * hidden_dim)
    self.lstm = torch.nn.LSTM(embedding_dim,hidden_dim ,num_layers = num_layers, batch_first = True,bidirectional=True) # O/P (batch,seq,2*features or hidden_dim)
  def forward(self,x,length):
    embed = self.embedding(x)
    padded_sequences = pack_padded_sequence(embed,length,batch_first = True,enforce_sorted = False)
    pad , (h_n,c_n) = self.lstm(padded_sequences)
    out , len = pad_packed_sequence(pad,batch_first = True) # out : (batch, seq_len, 2*hidden_dim)

    len = len.to('cuda')
    mask = torch.arange(out.size(1),device = 'cuda') < len[:,None]

    attn_wt = self.attention(out)   # attn_wt: (batch, seq_len, 2*hidden_dim)
    attn_wt = self.v(torch.tanh(attn_wt))   # attn_wt: (batch, seq_len, 1)

    attn_wt[~mask] = -float('inf')   # masking so that padding doesn't contribute to softmax probabilities and remain 0
    scores = torch.softmax(attn_wt,dim = 1)  # (batch, seq_len)

    dropout = self.dropout(torch.sum(scores*out,dim = 1))
    output = self.forward_layer1(dropout)  # Output is 1 to 3 sentiment

    return output


In [23]:
#weight = class_weight
#weight = (weight/weight.sum() )**1.5
#weight = weight.to('cuda')

model2 = Model2(len(vocabulary),embedding_dim = 200 ,hidden_dim = 128,num_layers=1, output_dim = 3 ,embedding_matrix = embedded_matrix)
criterion = torch.nn.CrossEntropyLoss()# Since 5 star rating examples are much more than 1 star , but already sampled with weightage , no need of extra penalty
optimizer = torch.optim.Adam(model2.parameters(),lr = 0.001, weight_decay = 1e-5)
model2 = model2.to('cuda')  # Move model to GPU

In [25]:
def training(epochs):
  for epoch in range(epochs):
    model2.train()
    epoch_loss = 0
    correct_pred_train = 0
    correct_pred_val = 0
    val_loss = 0

    for i,(x,y) in enumerate(train_loader):
      x,y = x.to('cuda'),y.to('cuda')
      length = (x!=0).sum(dim=1).cpu()   #sentences length are variable in each batch since they are shuffled too , hence runtime calculation req
      length[length==0] = 1
      length.cpu()
      optimizer.zero_grad()
      predictions = model2(x,length)
      y = y-1
      loss = criterion(predictions,y)
      epoch_loss += loss.item() # converting each loss to number
      loss.backward()   # Doing gradient of loss
      optimizer.step()  # Changing parameters based on derivative
      correct_pred_train += (torch.max(predictions,1)[1] == y).sum().item()   # summing total correct and converting to no

      #print(torch.bincount(torch.max(predictions,1)[1]))

      #torch.max(predictions,1) give (no_of_sentences,classes) and we retrieve highest logit,class_label tuple , but we r interested in label

    train_accuracy = correct_pred_train/len(train_loader.dataset)
    model2.eval()  #Set to evaluation mode
    with torch.no_grad():
      for val_x,val_y in (valid_loader):
        val_x,val_y = val_x.to('cuda'),val_y.to('cuda')
        length = (val_x!=0).sum(dim=1).cpu()
        length[length==0] = 1
        length.cpu()
        val_y = val_y - 1
        predictions = model2(val_x,length)
        loss = criterion(predictions,val_y)
        val_loss += loss.item()
        correct_pred_val += (torch.max(predictions,1)[1] == val_y).sum().item()

    val_accuracy = correct_pred_val/len(valid_loader.dataset)

    print(f"Epoch:{epoch+1}/{epochs} Training Loss : {epoch_loss} Accuracy : {train_accuracy} Validation Loss : {val_loss} Val accuracy : {val_accuracy}")

training(10)

Epoch:1/10 Training Loss : 1071.7679922766984 Accuracy : 0.8882204063923639 Validation Loss : 229.92612452805042 Val accuracy : 0.8893388225236778
Epoch:2/10 Training Loss : 1035.1009135954082 Accuracy : 0.8917347602863548 Validation Loss : 228.86049414426088 Val accuracy : 0.8892038424332411
Epoch:3/10 Training Loss : 1005.3883352577686 Accuracy : 0.8946031286909152 Validation Loss : 225.67331781238317 Val accuracy : 0.890823603518481
Epoch:4/10 Training Loss : 975.4737684540451 Accuracy : 0.8970617301805385 Validation Loss : 223.49091790616512 Val accuracy : 0.8915659940158827
Epoch:5/10 Training Loss : 951.4692838937044 Accuracy : 0.899939740159568 Validation Loss : 225.73162738233805 Val accuracy : 0.8912060471080515
Epoch:6/10 Training Loss : 924.7383978515863 Accuracy : 0.9021862270108709 Validation Loss : 223.2517656236887 Val accuracy : 0.8915659940158827
Epoch:7/10 Training Loss : 897.7091338150203 Accuracy : 0.9052811724154555 Validation Loss : 225.40374756604433 Val accuracy

# Attention Mechanism for Sentiment Analysis

## 1. Motivation: Why Attention?

- **Standard BiLSTM models** often rely on the last hidden state or Global Pooling (Mean/Max).
- **Limitation:** Pooling treats all words as equally important or only captures extreme values.
- **Goal:** Implement a mechanism that allows the model to *"focus"* on sentiment-heavy words (e.g., *"brilliant"*, *"disappointing"*, *"not"*) while ignoring noise.

---

## 2. Architectures Explored

### A. Linear Attention (Dot-Product Style)
- A simplified approach where importance is learned via a single weight matrix.
- **Formula:**  
  $$ e_i = W^T h_i $$
- **Observation:** Efficient and fast, but lacks the ability to capture complex non-linear relationships between words.

### B. Non-Linear Attention (Multi-Layer Perceptron Style)
- A more robust approach using a hidden layer and a non-linear activation function.
- **Formula:**  
  $$ e_i = v^T \tanh(W h_i + b) $$
- **Observation:** Using \(\tanh\), the model can better project the BiLSTM outputs into a space where importance is more easily separable.  
- **Note:** This was the primary architecture used for final results.

---

## 3. The Masking Mechanism (Critical Implementation)

- To handle variable-length sequences in batches:
  - A Boolean Mask was applied before the Softmax layer.
  - **Procedure:**
    - Identify padding tokens (indices \(\geq\) original length).
    - Set their attention scores to \(-\infty\).
  - **Result:** Softmax output for padding tokens becomes exactly **0**.

---

## 4. The "Sentiment Shift" (Problem Reframing)

- Initial experiments on **5-class ratings** faced a performance ceiling due to **Label Noise**:
  - Subjectivity between a 4-star and 5-star review.
- **Star Rating to Sentiment Labels:**
  - **1 & 2**: Negative (Strong negative sentiment)
  - **3**: Neutral (Mixed or indifferent)
  - **4 & 5**: Positive (Strong positive sentiment)
- **Impact:**  
  - This shift reduced the model's confusion, enabling the Attention mechanism to more clearly identify patterns associated with three distinct polarities.

---

## 5. Training Strategy & Results

- **Architecture:** BiLSTM + Non-Linear (tanh) Attention.
- **Handling Class Imbalance:**  
  - Natural distribution outperformed WeightedRandomSampler and manual loss weighting, which previously caused training instability.
- **Performance:**
  - **5-Class Accuracy:** approximately $68\%$ to $70\%$
  - **3-Class (Sentiment) Accuracy:** approximately $79\%$ to $80\%$

---

## 6. Core Insights

- **Attention = Learned Pooling:**  
  - Provides a *"weighted summary"* of the sentence that is more representative than simple averaging but in Attention too , words are treated independently and it doesn't learn difference between "not good" vs "good" , it sees each word indepdently unlike Q,K,V attention approach.
- **Task Complexity vs. Model Power:**  
  - Even with advanced Attention, the model's performance is limited by label quality.
  - Reducing from 5 to 3 classes significantly improved generalization.
- **Validation Integrity:**  
  - Keeping the validation set reflective of real-world data (no oversampling) ensured the roughly $80\%$ accuracy was a reliable metric.